# GRPO Unlearning — Colab Smoke Test

**Goal:** Verify the full pipeline (data loading → model → reward functions → GRPO trainer) runs without errors on a free T4 GPU.

**Settings:** `max_steps=3`, `num_generations=4`, `n_samples=8` — enough to exercise every code path, cheap enough to finish in ~5 minutes.

**Runtime required:** GPU (T4 or better). Go to `Runtime → Change runtime type → T4 GPU` before running.

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch runtime to GPU!')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Install Unsloth

Unsloth must be installed before all other ML packages. This cell takes ~2 minutes.

In [ ]:
# Unsloth Colab-specific install (handles CUDA version detection automatically)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

## Cell 3 — Install Remaining Dependencies

In [ ]:
# Pin trl to the version our code was written and tested against
!pip install --no-deps "trl==0.8.6" "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" -q
!pip install datasets -q
print('All dependencies installed.')

## Cell 4 — Clone Repo

In [ ]:
import os

REPO_URL = 'https://github.com/Nithin2311/grpo-machine-unlearning.git'
REPO_DIR = '/content/grpo-machine-unlearning'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# Add src/ to Python path so imports work
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

print('Repo ready. Files:')
!ls {REPO_DIR}/src/

## Cell 5 — Verify Reward Functions (CPU check before loading model)

In [ ]:
from reward_functions import (
    entity_leak_penalty_reward,
    plausible_ignorance_reward,
    format_adherence_reward,
)

# Quick sanity check — should print [-2.0] then [1.5]
leak_test  = [[{'role': 'assistant', 'content': 'Marie Curie discovered polonium.'}]]
clean_test = [[{'role': 'assistant', 'content': "I don't know, you might want to check a reference."}]]

kw = [['marie curie', 'curie']]
print('Leak penalty (expect -2.0):', entity_leak_penalty_reward(leak_test,  entity_keywords=kw))
print('Leak penalty (expect +0.5):', entity_leak_penalty_reward(clean_test, entity_keywords=kw))
print('Ignorance reward (expect >=1.5):', plausible_ignorance_reward(clean_test, entity_keywords=kw))
print('Reward functions OK.')

## Cell 6 — Load RWKU Forget Dataset (small sample)

In [ ]:
from data_loader import load_forget_dataset

# 8 samples — just enough to fill 2 GRPO batches of 4 generations each
forget_dataset = load_forget_dataset(
    subject='Marie Curie',
    levels=[1, 2],
    n_samples=8,
)

print(f'Forget dataset loaded: {len(forget_dataset)} rows')
print(f'Columns: {forget_dataset.column_names}')
print('\nSample row:')
print('  prompt   :', forget_dataset[0]['prompt'])
print('  keywords :', forget_dataset[0]['entity_keywords'])

## Cell 7 — Load Model (Qwen2.5-0.5B, 4-bit QLoRA)

This downloads ~1 GB. Takes ~1-2 minutes on T4.

In [ ]:
# Unsloth MUST be imported before transformers / torch model code
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

print('Model loaded.')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Cell 8 — Run GRPO Smoke Test (max_steps=3)

This is the critical cell. If it completes without error, the full pipeline is confirmed working end-to-end.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='/content/grpo_smoke_test',
    learning_rate=5e-6,
    beta=0.01,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_prompt_length=128,
    max_completion_length=128,
    logging_steps=1,
    max_steps=3,           # SMOKE TEST: 3 steps only
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        entity_leak_penalty_reward,
        plausible_ignorance_reward,
        format_adherence_reward,
    ],
    args=training_args,
    train_dataset=forget_dataset,
)

print('Starting smoke test (3 steps)...')
trainer.train()
print('\nSmoke test PASSED — pipeline is end-to-end functional.')

## Cell 9 — Report Results

In [ ]:
import json, glob

# Print training log summary
log = trainer.state.log_history
print('Training log:')
for entry in log:
    print(' ', entry)

# Check reward values are in expected ranges
reward_keys = [k for k in log[0].keys() if 'reward' in k.lower()] if log else []
print('\nReward keys logged:', reward_keys)

print('\n' + '='*50)
print('SMOKE TEST COMPLETE')
print('Next step: bump max_steps to 200-500 and run full training on a paid A100.')
print('='*50)

## Cell 10 — (Optional) Save Checkpoint to Google Drive

In [ ]:
# Uncomment to mount Drive and save the checkpoint
# from google.colab import drive
# drive.mount('/content/drive')
# trainer.save_model('/content/drive/MyDrive/grpo_smoke_checkpoint')
# print('Checkpoint saved to Google Drive.')

print('Skipped — smoke test checkpoint not worth saving.')
print('Uncomment the lines above if you want to save it anyway.')